# Kaggle Test Set Prediction

**Objective:** Load the labelled training data prepared in Notebook 2, apply the same classification preprocessing to Kaggle's unlabelled test data, train the selected TF-IDF model on all labelled rows, and create `submission.csv`.

The Kaggle test set is not part of `train_dataset_preprocessed.pkl`; this notebook loads it separately from `Datasets/test.csv`. Test IDs are preserved and aligned to `sample_submission.csv` when writing the output.

## 1. Load the prepared training data and Kaggle test data

Notebook 2 saved the processed, labelled training rows. The Kaggle test CSV contains the test lyrics and IDs, but no genre labels.

In [16]:
import pandas as pd
import numpy as np
import pickle
from sklearn.base import clone

from sklearn.pipeline import FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import classification_report, f1_score

In [17]:
with open('test_for_classification.pkl', 'rb') as f:
    test_for_classification = pickle.load(f)

print("Test dataset loaded successfully!")

Test dataset loaded successfully!


In [18]:
with open('train_for_classification.pkl', 'rb') as f:
    train_for_classification = pickle.load(f)

print("Train dataset loaded successfully!")

Train dataset loaded successfully!


In [19]:
train_for_classification.shape

(134776, 10)

## 2. Apply the classification preprocessing to Kaggle test lyrics

Use the same regex rules as Notebook 2, then the shared tokenization, stopword-removal, and lemmatization pipeline. Do not remove any test rows: Kaggle expects one prediction for every test ID.

In [13]:
test_for_classification

,title,artist,year,views,features,raw_lyrics,lyrics_cleaned,lyrics_normalized
id,,,,,,,,
3996359,Like You Do,Jimmie Allen,2018,825,{},[Verse 1]\r\nYour head's on my chest\r\nThis b...,Your head's on my chest This bed's all a mess ...,head chest bed mess empty bottle red wine tell...
3553166,This Building is Falling Down,The Taxpayers,2009,400,{},Epilogue: A sound off from simple people with ...,Epilogue A sound off from simple people with s...,epilogue sound simple people simple theory fuc...
2998742,My Private Joy,Dion and The Belmonts,1960,229,{},[Verse 1]\r\nOh well Susie is a doozy with a p...,Oh well Susie is a doozy with a pony tail She ...,oh well susie doozy pony tail wear gold polish...
3993470,Vegan,Yung Kurry,2018,1041,{},Hook\r\nShe a vegan but she eat my meat with n...,Hook She a vegan but she eat my meat with no s...,vegan eat meat season take look wrist ooh shit...
5488256,To The Boy I Used To Know,Angelina Cruz,2018,63,{},To the boy I used to know\r\nI wonder how do y...,To the boy I used to know I wonder how do you ...,boy use wonder boy use still think old break m...
...,...,...,...,...,...,...,...,...
1149534,My Life With You,Celtic Thunder,2015,32,{},When I was a younger man\r\nWhen I was a silly...,When I was a younger man When I was a silly bo...,young man silly boy need thing strong anything...
2473358,Work it out,The Jacka,2014,488,{Husalah},[Intro]\r\nNigga did you take my money?\r\nHeh...,Nigga did you take my money Hehehe that's righ...,nigga take money hehehe right lil bitch tryna ...
5254514,No More,Darling West,2016,20,{},The moon is hanging high above\r\nA long and d...,The moon is hanging high above A long and dust...,moon hang high long dusty red dirt road blood ...


In [14]:
import pandas as pd
from sklearn.pipeline import FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier

# 1. Preparar os dados de treino completos e o conjunto de teste do Kaggle
X_train_full = train_for_classification["lyrics_normalized"].fillna("").astype(str)
y_train_full = train_for_classification["tag"]
X_test = test_for_classification["lyrics_normalized"].fillna("").astype(str)

# 2. Configurar o FeatureUnion vencedor (Word 1-2 + Char_wb 3-5)
tfidf_vectorizer = FeatureUnion([
    ('word', TfidfVectorizer(
        max_df=0.7,
        min_df=5,
        max_features=75000,
        ngram_range=(1, 2),
        sublinear_tf=True
    )),
    ('char', TfidfVectorizer(
        analyzer='char_wb',
        ngram_range=(3, 5),
        min_df=5,
        max_features=75000,
        sublinear_tf=True
    ))
])

print("A vetorizar os dados de treino (Word + Char_wb)...")
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train_full)

print("A vetorizar os dados de teste...")
X_test_tfidf = tfidf_vectorizer.transform(X_test)

# 3. Configurar o SGDClassifier com os parâmetros exatos do 0.5222
final_model = SGDClassifier(
    loss="log_loss",
    alpha=3e-05,
    class_weight="balanced",
    max_iter=1000,
    random_state=42
)

print("A treinar o modelo campeão nos 100% do treino...")
final_model.fit(X_train_tfidf, y_train_full)

# 4. Gerar previsões finais para o Kaggle
test_predictions = final_model.predict(X_test_tfidf)

print("Previsões criadas com sucesso:", len(test_predictions))
print("\nDistribuição das previsões no Kaggle Test (%):")
print((pd.Series(test_predictions).value_counts(normalize=True) * 100).round(2))

A vetorizar os dados de treino (Word + Char_wb)...
A vetorizar os dados de teste...
A treinar o modelo campeão nos 100% do treino...
Previsões criadas com sucesso: 33742

Distribuição das previsões no Kaggle Test (%):
pop        39.04
rap        28.75
rock       16.31
rb          5.91
misc        5.23
country     4.74
Name: proportion, dtype: float64


In [20]:
import pandas as pd
from sklearn.pipeline import FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier

# 1. Preparar os dados de treino completos e o conjunto de teste do Kaggle
X_train_full = train_for_classification["lyrics_normalized"].fillna("").astype(str)
y_train_full = train_for_classification["tag"]
X_test = test_for_classification["lyrics_normalized"].fillna("").astype(str)


tfidf_vectorizer = TfidfVectorizer(
        max_df=0.75,
        min_df=5,
        max_features=75000,
        ngram_range=(1, 2),
        sublinear_tf=True
    )


print("A vetorizar os dados de treino ...")
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train_full)

print("A vetorizar os dados de teste...")
X_test_tfidf = tfidf_vectorizer.transform(X_test)

# 3. Configurar o SGDClassifier com os parâmetros exatos do 0.5222
final_model = SGDClassifier(
    loss="log_loss",
    alpha=3e-05,
    class_weight="balanced",
    max_iter=1000,
    random_state=42
)

print("A treinar o modelo campeão nos 100% do treino...")
final_model.fit(X_train_tfidf, y_train_full)

# 4. Gerar previsões finais para o Kaggle
test_predictions = final_model.predict(X_test_tfidf)

print("Previsões criadas com sucesso:", len(test_predictions))
print("\nDistribuição das previsões no Kaggle Test (%):")
print((pd.Series(test_predictions).value_counts(normalize=True) * 100).round(2))

A vetorizar os dados de treino (Word + Char_wb)...
A vetorizar os dados de teste...
A treinar o modelo campeão nos 100% do treino...
Previsões criadas com sucesso: 33742

Distribuição das previsões no Kaggle Test (%):
pop        42.63
rap        28.61
rock       14.15
rb          5.57
misc        5.15
country     3.88
Name: proportion, dtype: float64


In [7]:
with open("best_logistic_pipeline.pkl", "rb") as f:
    best_model = pickle.load(f)

FileNotFoundError: [Errno 2] No such file or directory: 'best_logistic_pipeline.pkl'

## 4. Create and validate the Kaggle submission

Use the IDs and column order from `sample_submission.csv` so the output matches Kaggle's expected format.

In [14]:
with open("best_sgd_pipeline.pkl", "rb") as f:
    best_sgd = pickle.load(f)

In [18]:
with open("best_word_char_sgd.pkl", "rb") as f:
    best_word_char_sgd = pickle.load(f)

In [ ]:
with open("word_char_sgd_offsets.pkl", "rb") as f:
    best_word_char_sgd_offsets = pickle.load(f)

In [23]:
best_model = best_word_char_sgd_offsets["pipeline"]
offsets = best_word_char_sgd_offsets["offsets"]
score_scales = best_word_char_sgd_offsets["score_scales"]
classes = best_word_char_sgd_offsets["classes"]

In [24]:
X_train_full = train_for_classification["lyrics_normalized"]
y_train_full = train_for_classification["tag"]

X_test = test_for_classification["lyrics_normalized"]

final_model = clone(best_word_char_sgd)
final_model.fit(X_train_full, y_train_full)

# Verify class order
assert np.array_equal(
    final_model.classes_, classes
)

# Get decision scores
test_scores = final_model.decision_function(X_test)

# Apply the optimized offsets
adjusted_scores = test_scores + offsets * score_scales

# Predict on Kaggle test
test_predictions = classes[np.argmax(adjusted_scores, axis=1)]


print("Model:", final_model.named_steps["classifier"])
print("Total predictions:", len(test_predictions))
print("Offsets:", dict(zip(classes, offsets)))

Model: SGDClassifier(class_weight='balanced', loss='log_loss', random_state=42)
Total predictions: 33742
Offsets: {'country': -0.25, 'misc': -0.5, 'pop': 0.0, 'rap': 0.0, 'rb': 0.0, 'rock': 0.25}


In [21]:
sample_submission = pd.read_csv("Datasets/test.csv")[["id"]]

predictions_by_id = pd.Series(test_predictions, index=test_for_classification.index, name="tag")
submission = (
    predictions_by_id.reindex(sample_submission["id"])
    .rename_axis("id")
    .reset_index()
)

assert len(submission) == len(sample_submission)
assert submission["tag"].notna().all()
assert submission.columns.tolist() == ["id", "tag"]
assert submission["id"].equals(sample_submission["id"])

submission.to_csv("submission.csv", index=False)
print("Saved submission.csv")
display(submission.head())

Saved submission.csv


,id,tag
0,3996359,rb
1,3553166,rock
2,2998742,pop
3,3993470,rap
4,5488256,pop
